# Regression on house prices - advanced

I will predict the impact probability based of an asteroid based on its characteristics.
TODO: explain the advanced methods used

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display
from pandas.plotting import scatter_matrix
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler, FunctionTransformer

### 1. Data preparation
I will reuse some of the findings from my Data Exploration Advanced notebook, where I used the exact same dataset as in this one.

In [ ]:
impacts = pd.read_csv("impacts.csv")
impacts["Period Length"] = impacts["Period End"] - impacts["Period Start"]

I added a new column to the dataset, which measures how long the period lasts for each asteroid. This might correlate with impact probability or number of impacts, because it has a longer time window where it can hit.

In [ ]:
impacts.info()
impacts.describe()

In [ ]:
impacts["Maximum Torino Scale"].value_counts()

Since `Maximum Torino Scale` is all zeros except 3 invalid `(*)` values, we will drop the columns in the Pipeline. It's not useful for linear regression. As seen in my advanced data exploration notebook, all other columns are clean, they don't have any invalid/null values.

In [ ]:
impacts.hist(bins=50, figsize=(20, 15))
plt.show()

Drawing histograms of all features to get a general understanding of how the data is distributed. We can see that some of them are very heavily skewed right.

In [ ]:
columns = ["Cumulative Impact Probability", "Possible Impacts", "Asteroid Diameter (km)"]
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

for ax, column in zip(axes, columns):
    ax.hist(np.log10(impacts[column]), bins=50)
    ax.set(xlabel=f"log10({column})", ylabel="Number of asteroids")

plt.show()

Log-plots of some heavily right skewed distributions.

In [ ]:
scatter_matrix(impacts, figsize=(30, 30))
plt.show()

Scatter plot of each feature against each feature. On the diagonal, there is a histogram for every feature. A correlation matrix is displayed below.





In [ ]:
corr = impacts.corr(numeric_only=True)

plt.figure(figsize=(10, 8))
sns.heatmap(
    corr,
    annot=True,  # print numbers inside cells
    fmt=".2f",  # two decimal places
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    center=0,
)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


The correlation matrix shows weak positive linear associations between raw impact probability and the Palermo scales. Period Length is moderately correlated with Possible Impacts, but has almost no linear association with raw impact probability. Since several variables are highly skewed, we will also examine correlations after log transformations. We will then use cross-validation to assess whether multiple predictors or interaction terms improve prediction.

# 2. Training the model and evaluation

Let's prepare the data transformation pipeline. We will also use log10 to transform some of the features with skewed distributions. We will also use GridSearchCV to go through all hyperparameter combinations and choose the best one. At the end, we will measure RMSE over a test set, which wasn't used to train the model.

In [ ]:
log_columns = ["Possible Impacts", "Asteroid Diameter (km)"]
common = [StandardScaler()]
log_transformer = FunctionTransformer(np.log10)

preprocessing = ColumnTransformer([
    ("text", "drop", make_column_selector(dtype_include=str)),
    ("log", make_pipeline(log_transformer, *common), log_columns),
], remainder=make_pipeline(*common))
pipeline = Pipeline([
    ("preprocess", preprocessing),
    ("polynomial", PolynomialFeatures()),
    ("regression", LinearRegression())
])

param_grid = {
    "polynomial__degree": [0, 1, 2, 3],
    "preprocess__log": [
        make_pipeline(*common),
        make_pipeline(log_transformer, *common),
    ],
}
grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="neg_root_mean_squared_error",
)

X = impacts.drop(columns=["Cumulative Impact Probability"])
y = np.log10(impacts["Cumulative Impact Probability"])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

_ = grid_search.fit(X_train, y_train)

In [ ]:
best_score = grid_search.score(X_test, y_test)

print("Best params: ", grid_search.best_params_)
print("Best score (unit is 1 order of magnitude): ", f"{best_score} ({best_score:e})")
print(f"The average error is {10**best_score} percentage points")

In [ ]:
pd.DataFrame(grid_search.cv_results_).drop(columns=["mean_fit_time", "std_fit_time", "mean_score_time", "std_score_time"]).sort_values("rank_test_score")

It turns out that doing log10 of skewed features lead to a worse RMSE score, because grid search chose the model without it.

In [ ]:
target = "Cumulative Impact Probability"
X = impacts.drop(columns=[target])
y = np.log10(impacts[target])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

grid_search.fit(X_train, y_train)
y_pred = grid_search.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

low = min(y_test.min(), y_pred.min())
high = max(y_test.max(), y_pred.max())

axes[0].scatter(y_test, y_pred, alpha=0.7)
axes[0].plot([low, high], [low, high], "r--")
axes[0].set(
    xlabel="Actual log10(probability)",
    ylabel="Predicted log10(probability)",
    title="Actual vs. predicted",
)

residuals = y_test - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.7)
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set(
    xlabel="Predicted log10(probability)",
    ylabel="Residual (actual − predicted)",
    title="Residuals vs. predicted",
)

plt.tight_layout()
plt.show()